In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load data [cite: 26]
fraud_df = pd.read_csv('../data/raw/Fraud_Data.csv')

# --- Data Cleaning ---
fraud_df.dropna(inplace=True) # Dropping missing values to maintain data integrity [cite: 108]
fraud_df.drop_duplicates(inplace=True) # Removing duplicates [cite: 109]
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time']) # Correcting types [cite: 110]
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])

# --- Geolocation Integration ---
ip_df = pd.read_csv('../data/raw/IpAddress_to_Country.csv')
fraud_df['ip_address'] = fraud_df['ip_address'].astype(float)
fraud_df = fraud_df.sort_values('ip_address')
ip_df = ip_df.sort_values('lower_bound_ip_address')

# Range-based lookup [cite: 117]
merged_df = pd.merge_asof(fraud_df, ip_df, left_on='ip_address', right_on='lower_bound_ip_address', direction='backward')
merged_df.loc[merged_df['ip_address'] > merged_df['upper_bound_ip_address'], 'country'] = 'Unknown'

In [ ]:
# --- 4. Univariate Analysis ---
plt.figure(figsize=(10, 5))
sns.histplot(fraud_df['purchase_value'], bins=50, kde=True, color='blue')
plt.title("Univariate: Distribution of Purchase Values (E-commerce)")
plt.xlabel("Purchase Value ($)")
plt.ylabel("Frequency")
plt.show()


In [ ]:

# --- 5. Bivariate Analysis ---
plt.figure(figsize=(10, 5))
sns.boxplot(data=fraud_df, x='class', y='purchase_value', palette='Set2')
plt.title("Bivariate: Purchase Value Distribution by Class")
plt.xlabel("Class (0 = Legitimate, 1 = Fraud)")
plt.ylabel("Purchase Value ($)")
plt.show()

In [ ]:
# --- 6. Geolocation Integration ---
# Load the IP mapping dataset
ip_df = pd.read_csv('../data/raw/IpAddress_to_Country.csv')
ip_df.dropna(inplace=True)

# Convert IP addresses to float for numerical range mapping
fraud_df['ip_address'] = fraud_df['ip_address'].astype(float)

# Sort both dataframes by the IP columns for merge_asof to work correctly
fraud_df = fraud_df.sort_values('ip_address')
ip_df = ip_df.sort_values('lower_bound_ip_address')

# Merge using range-based lookup
merged_df = pd.merge_asof(fraud_df, ip_df, 
                          left_on='ip_address', 
                          right_on='lower_bound_ip_address', 
                          direction='backward')

# Clean up the merge: if an IP exceeded the upper bound, it doesn't belong to that country
merged_df.loc[merged_df['ip_address'] > merged_df['upper_bound_ip_address'], 'country'] = 'Unknown'
merged_df['country'] = merged_df['country'].fillna('Unknown')

print(f"Geolocation integration complete. Merged Data Shape: {merged_df.shape}")

In [ ]:
# --- 7. Analyze Fraud Patterns by Country ---
# Group by country to calculate total transactions and total fraud cases
country_stats = merged_df.groupby('country').agg(
    total_transactions=('class', 'count'),
    fraud_cases=('class', 'sum')
).reset_index()

# Calculate the fraud rate as a percentage
country_stats['fraud_rate'] = (country_stats['fraud_cases'] / country_stats['total_transactions']) * 100

# Filter out 'Unknown' to focus on actual countries, then grab the top 10 by fraud volume
top_fraud_countries = country_stats[country_stats['country'] != 'Unknown'].sort_values(by='fraud_cases', ascending=False).head(10)

print("\n--- Top 10 Countries by Total Fraud Cases ---")
print(top_fraud_countries.to_string(index=False))

# Visualization for your report
plt.figure(figsize=(12, 6))
sns.barplot(data=top_fraud_countries, x='fraud_cases', y='country', palette='Reds_r')
plt.title("Top 10 Countries with the Highest Number of Fraud Cases")
plt.xlabel("Total Fraud Cases")
plt.ylabel("Country")
plt.show()